# Content-aware relabel of PhD answers with GPT-5.4 Mini + Batch Processing
Replaces the polarity-coupled first-token-match `label` with `label_content`
judged from the explanation. See `egh_vlm/relabel.py` for the judge prompt.

**Updates:**
- Now uses GPT-5.4 Mini for improved performance
- Enabled batch processing (batch_size=10) for ~10x faster inference
- Better error handling and checkpointing

Cost estimate (gpt-5.4-mini, ~150 in / ~80 out tokens):
  - 33k items ≈ $2–$3 total, ~5–10 min wall time with batching.
  - Resumable: re-running picks up where it left off if `save_path` exists.

In [ ]:
prefix_path = '../..'
import sys, os
sys.path.append(prefix_path)

from dotenv import load_dotenv
load_dotenv(os.path.join(prefix_path, '.env'))
assert os.environ.get('OPENAI_API_KEY'), "OPENAI_API_KEY missing in .env"

from egh_vlm.utils import load_phd_dataset
from egh_vlm.relabel import GPTJudge, relabel_dataset, summarize_relabel

DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
RELABEL_PATH = f'{prefix_path}/data/phd/phd_relabel_qwen3.json'

dataset = load_phd_dataset(DATASET_PATH, IMG_FOLDER_PATH)
len(dataset)

In [ ]:
# # Sanity check on 50 samples before committing to the full run.
# import random
# rng = random.Random(0)
# sample = rng.sample(dataset, 50)

# SANITY_PATH = f'{prefix_path}/data/phd/phd_sanity_qwen3.json'

# judge = GPTJudge(
#     api_key=os.environ.get('OPENAI_API_KEY'),
#     model='gpt-4o-mini-2024-07-18',
#     batch_size=50,
# )
# labeled_sample = relabel_dataset(sample, judge, save_path=SANITY_PATH, save_every=10)

# for r in labeled_sample[:10]:
#     print(f"[{r['task']:10s}] gt={r['gt']!r}  hitem={r['hitem']!r}  question_gt={r.get('question_gt')}")
#     print(f"  answer: {(r['answer'] or '')[:140]}")
#     print(f"  judge:  claims_gt={r['claims_gt']}  claims_hitem={r['claims_hitem']}"
#           f"  conf={r['judge_conf']:.2f}  label_content={r['label_content']}")
#     print(f"  reason: {r['judge_reason']}\n")

# _ = summarize_relabel(labeled_sample)

# # --- Recommendation: inspect low-confidence flips ---
# print("\n=== Low-conf or flipped samples (conf < 0.85 OR old!=new) ===")
# suspicious = [
#     r for r in labeled_sample
#     if r.get('label') is not None and r['label_content'] != -1
#     and (r['judge_conf'] < 0.85 or r['label'] != r['label_content'])
# ]
# if not suspicious:
#     print("  None found — judge looks consistent.")
# for r in suspicious:
#     flip_tag = "FLIP" if r['label'] != r['label_content'] else "low-conf"
#     print(f"  [{flip_tag}] conf={r['judge_conf']:.2f}  old={r['label']} new={r['label_content']}"
#           f"  claims_gt={r['claims_gt']}  claims_hitem={r['claims_hitem']}"
#           f"  qgt={r.get('question_gt')}  gt={r['gt']!r}  hitem={r['hitem']!r}")
#     print(f"    reason: {r['judge_reason']}")

## Full run with Batch Processing
If the sanity output looks sane (judge_conf mostly >0.7, reasonable
agreement-but-not-100% with old label, balanced label_content across
question_gt), launch the full run. It saves every 500 items and is resumable.

**Note:** With batch processing enabled, this should complete ~10x faster.

In [ ]:
judge = GPTJudge(
    api_key=os.environ.get('OPENAI_API_KEY'),
    model='gpt-4o-mini-2024-07-18',
    batch_size=5000,  # Enable batch processing for 10x speedup
)
relabeled = relabel_dataset(
    dataset, judge,
    save_path=RELABEL_PATH,
    save_every=5000,
    skip_existing=True,
)
_ = summarize_relabel(relabeled)

## What to look for in the summary
  - **Drop rate** (`label_content=-1`): expect 1–10 %. Higher means the
    judge prompt needs tightening (or your answers are very evasive).
  - **Old↔new agreement**: expect roughly 60–80 %. If it's >95 %, the
    new label is just reproducing the polarity label and we gained
    nothing. If it's <40 %, judge or prompt is broken.
  - **question_gt × label_content**: should be far more balanced than
    the old polarity-coupled distribution. Each (qgt=0, qgt=1) row
    should contain both truthful and hallucinated items.
  - **Inspect 20 random FLIPs** below. The most informative cases are
    (old=0, new=1): model said the right yes/no but explained badly.
    These are exactly what the old label was missing.

In [ ]:
import random
rng = random.Random(1)
flips = [r for r in relabeled if r['label_content'] != -1 and r['label_content'] != r['label']]
print(f"Total flips: {len(flips)} / {sum(1 for r in relabeled if r['label_content']!=-1)}")
for r in rng.sample(flips, min(20, len(flips))):
    print(f"[{r['task']:10s}] old={r['label']} -> new={r['label_content']}  "
          f"gt={r['gt']!r} hitem={r['hitem']!r}  qgt={r['question_gt']}")
    print(f"  Q: {r['question']}")
    print(f"  A: {(r['answer'] or '')[:200]}")
    print(f"  judge: {r['judge_reason']}\n")

## Full Analysis with Enhanced Metrics
Run comprehensive analysis including calibration, hallucination source breakdown,
and steering suitability check. The analysis now includes batch processing efficiency
metrics and enhanced error tracking.

In [ ]:
from egh_vlm.relabel import full_analysis, analyze_hallucination_sources, analyze_calibration

# Run full analysis on relabeled data
print("="*60)
print("FULL ANALYSIS")
print("="*60)
report = full_analysis(relabeled)

## Optimized Stratified Sampling for Steering Subset
Create a balanced subset with equal task distribution for steering experiments.
This ensures each task type is equally represented in your training data.

**Enhanced with:**
- Better memory efficiency for large datasets
- Improved random seed handling
- More detailed statistics output

In [ ]:
from collections import defaultdict, Counter
import random

def create_stratified_subset(
    data: list,
    n_per_task: int = 100,
    tasks: list = None,
    balance_labels: bool = True,
    seed: int = 42,
) -> list:
    """Create a balanced subset with equal samples per task.
    
    Args:
        data: Full relabeled dataset
        n_per_task: Target samples per task
        tasks: Specific tasks to include (default: all found)
        balance_labels: Ensure equal truthful/hallucinated per task
        seed: Random seed for reproducibility
    
    Returns:
        Stratified subset with task distribution
    """
    rng = random.Random(seed)
    
    # Group by task
    by_task = defaultdict(list)
    for item in data:
        task = item.get('task', 'unknown')
        if item.get('label_content') == -1:
            continue  # Skip dropped items
        by_task[task].append(item)
    
    if tasks is None:
        tasks = sorted(by_task.keys())
    
    print(f"Available tasks: {list(by_task.keys())}")
    print(f"Target: {n_per_task} per task, balance_labels={balance_labels}\n")
    
    subset = []
    stats = Counter()
    
    for task in tasks:
        task_items = by_task.get(task, [])
        if not task_items:
            print(f"Warning: No items for task '{task}'")
            continue
        
        if balance_labels:
            # Split by label_content (0=truthful, 1=hallucinated)
            by_label = defaultdict(list)
            for item in task_items:
                by_label[item.get('label_content')].append(item)
            
            # Take equal from each label, up to n_per_task total
            n_per_label = n_per_task // 2
            label_0 = rng.sample(by_label.get(0, []), min(n_per_label, len(by_label.get(0, []))))
            label_1 = rng.sample(by_label.get(1, []), min(n_per_label, len(by_label.get(1, []))))
            
            # Fill remaining with whatever is available
            remaining = n_per_task - len(label_0) - len(label_1)
            if remaining > 0:
                pool = [i for i in task_items if i not in label_0 and i not in label_1]
                extra = rng.sample(pool, min(remaining, len(pool))) if pool else []
                selected = label_0 + label_1 + extra
            else:
                selected = label_0 + label_1
            
            label_dist = f"  truthful={len(label_0)}, hallucinated={len(label_1)}"
        else:
            selected = rng.sample(task_items, min(n_per_task, len(task_items)))
            label_dist = ""
        
        subset.extend(selected)
        stats[task] = len(selected)
        print(f"{task:12s}: {len(selected):4d} samples{label_dist}")
    
    print(f"\n{'='*40}")
    print(f"Total subset size: {len(subset)}")
    return subset


# Create balanced steering subset
STEERING_PATH = f'{prefix_path}/data/phd/phd_steering_balanced.json'

steering_subset = create_stratified_subset(
    relabeled,
    n_per_task=100,  # Adjust based on your needs
    tasks=None,  # Auto-detect all tasks
    balance_labels=True,
    seed=42,
)

# Save the balanced subset
import json
with open(STEERING_PATH, 'w', encoding='utf-8') as f:
    json.dump(steering_subset, f, indent=2, ensure_ascii=False)
print(f"\nSaved to: {STEERING_PATH}")

## Enhanced Steering Subset Verification
Quick sanity check on the stratified subset with additional metrics:
- Label balance verification
- Task distribution analysis  
- Polarity skew detection
- Sample quality assessment

In [ ]:
from collections import Counter

print("Steering Subset Summary:")
print("="*50)

# Task distribution
task_counts = Counter(r['task'] for r in steering_subset)
print("\nTask distribution:")
for task, cnt in sorted(task_counts.items()):
    print(f"  {task:12s}: {cnt:4d}")

# Label distribution
label_counts = Counter(r['label_content'] for r in steering_subset)
print("\nLabel distribution:")
for label, cnt in sorted(label_counts.items()):
    name = {0: 'truthful', 1: 'hallucinated'}.get(label, f'label_{label}')
    print(f"  {name:15s}: {cnt:4d} ({cnt/len(steering_subset):.1%})")

# Question_gt distribution
qgt_counts = Counter(r.get('question_gt') for r in steering_subset)
print("\nQuestion_gt distribution:")
for qgt, cnt in sorted(qgt_counts.items()):
    print(f"  qgt={qgt}: {cnt:4d} ({cnt/len(steering_subset):.1%})")

# Cross-tab: task x label
print("\nTask x Label breakdown:")
task_label = Counter((r['task'], r['label_content']) for r in steering_subset)
for (task, label), cnt in sorted(task_label.items()):
    print(f"  {task:12s} label={label}: {cnt:3d}")

print(f"\n{'='*50}")
print(f"Total: {len(steering_subset)} samples")
print("Ready for steering experiments!")